<a href="https://colab.research.google.com/github/ThandoZwane06/Cisco-Data-Science-Essentials/blob/main/Lessons%20/%20Week_3_Lessons%20/%20SA_Fraud_LogRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving sa_fraud_transactions.csv to sa_fraud_transactions.csv


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


zar_fraud = pd.read_csv("sa_fraud_transactions.csv")
zar_fraud

,transaction_id,customer_id,timestamp,amount_zar,merchant_category,province,customer_home_province,is_fraud
0,TXN00001,CUST031,2026-01-01 02:24:15,1881.28,Clothing,Gauteng,Gauteng,1
1,TXN00002,CUST042,2026-01-02 02:50:18,2212.07,Online Retail,North West,North West,1
2,TXN00003,CUST016,2026-01-02 14:05:14,486.45,Fuel,Mpumalanga,Mpumalanga,0
3,TXN00004,CUST017,2026-01-02 16:07:43,2009.66,Travel,Limpopo,Limpopo,0
4,TXN00005,CUST025,2026-01-03 11:04:22,619.25,Clothing,Gauteng,Gauteng,0
...,...,...,...,...,...,...,...,...
355,TXN00356,CUST001,2026-03-30 19:34:02,517.01,Electronics,Gauteng,Gauteng,0
356,TXN00357,CUST027,2026-03-31 07:55:02,273.42,Restaurants,North West,North West,0
357,TXN00358,CUST020,2026-03-31 10:08:12,249.62,Entertainment,Mpumalanga,Mpumalanga,0
358,TXN00359,CUST020,2026-03-31 15:50:23,704.33,Fuel,Mpumalanga,Mpumalanga,0


In [4]:
zar_fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          360 non-null    object 
 1   customer_id             360 non-null    object 
 2   timestamp               360 non-null    object 
 3   amount_zar              360 non-null    float64
 4   merchant_category       360 non-null    object 
 5   province                360 non-null    object 
 6   customer_home_province  360 non-null    object 
 7   is_fraud                360 non-null    int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 22.6+ KB


In [5]:
zar_fraud['is_fraud'].value_counts()

,count
is_fraud,
0,324
1,36


In [6]:
zar_fraud.isnull().sum()

,0
transaction_id,0
customer_id,0
timestamp,0
amount_zar,0
merchant_category,0
province,0
customer_home_province,0
is_fraud,0


In [7]:
print((zar_fraud['is_fraud'].value_counts()[1] / zar_fraud.shape[0])*100)

10.0


In [8]:
#convert timestamp to datetime
zar_fraud['timestamp'] = pd.to_datetime(zar_fraud['timestamp'])

zar_fraud['hour'] = zar_fraud['timestamp'].dt.hour


In [9]:
#calculate the time between transactions
zar_fraud['period_between_transx'] = (zar_fraud.groupby('customer_id')['timestamp'].diff().dt.total_seconds()/60.0).round(0)

zar_fraud

,transaction_id,customer_id,timestamp,amount_zar,merchant_category,province,customer_home_province,is_fraud,hour,period_between_transx
0,TXN00001,CUST031,2026-01-01 02:24:15,1881.28,Clothing,Gauteng,Gauteng,1,2,NaN
1,TXN00002,CUST042,2026-01-02 02:50:18,2212.07,Online Retail,North West,North West,1,2,NaN
2,TXN00003,CUST016,2026-01-02 14:05:14,486.45,Fuel,Mpumalanga,Mpumalanga,0,14,NaN
3,TXN00004,CUST017,2026-01-02 16:07:43,2009.66,Travel,Limpopo,Limpopo,0,16,NaN
4,TXN00005,CUST025,2026-01-03 11:04:22,619.25,Clothing,Gauteng,Gauteng,0,11,NaN
...,...,...,...,...,...,...,...,...,...,...
355,TXN00356,CUST001,2026-03-30 19:34:02,517.01,Electronics,Gauteng,Gauteng,0,19,43666.0
356,TXN00357,CUST027,2026-03-31 07:55:02,273.42,Restaurants,North West,North West,0,7,28111.0
357,TXN00358,CUST020,2026-03-31 10:08:12,249.62,Entertainment,Mpumalanga,Mpumalanga,0,10,1068.0
358,TXN00359,CUST020,2026-03-31 15:50:23,704.33,Fuel,Mpumalanga,Mpumalanga,0,15,342.0


In [10]:
#customer average amount spend
cust_avg_spend = zar_fraud.groupby('customer_id')['amount_zar'].mean().round(2).reset_index()
cust_avg_spend = cust_avg_spend.rename(columns={'amount_zar':'cust_avg_amount'})


In [11]:
#Merge
zar_fraud = zar_fraud.merge(cust_avg_spend, on='customer_id', how='left')
zar_fraud

,transaction_id,customer_id,timestamp,amount_zar,merchant_category,province,customer_home_province,is_fraud,hour,period_between_transx,cust_avg_amount
0,TXN00001,CUST031,2026-01-01 02:24:15,1881.28,Clothing,Gauteng,Gauteng,1,2,NaN,998.13
1,TXN00002,CUST042,2026-01-02 02:50:18,2212.07,Online Retail,North West,North West,1,2,NaN,1065.60
2,TXN00003,CUST016,2026-01-02 14:05:14,486.45,Fuel,Mpumalanga,Mpumalanga,0,14,NaN,568.42
3,TXN00004,CUST017,2026-01-02 16:07:43,2009.66,Travel,Limpopo,Limpopo,0,16,NaN,811.80
4,TXN00005,CUST025,2026-01-03 11:04:22,619.25,Clothing,Gauteng,Gauteng,0,11,NaN,557.51
...,...,...,...,...,...,...,...,...,...,...,...
355,TXN00356,CUST001,2026-03-30 19:34:02,517.01,Electronics,Gauteng,Gauteng,0,19,43666.0,518.95
356,TXN00357,CUST027,2026-03-31 07:55:02,273.42,Restaurants,North West,North West,0,7,28111.0,843.84
357,TXN00358,CUST020,2026-03-31 10:08:12,249.62,Entertainment,Mpumalanga,Mpumalanga,0,10,1068.0,558.36
358,TXN00359,CUST020,2026-03-31 15:50:23,704.33,Fuel,Mpumalanga,Mpumalanga,0,15,342.0,558.36


In [12]:
# amount spent by customer vs their average
amount_vs_avg = (zar_fraud['amount_zar']/zar_fraud['cust_avg_amount']).round(2)

# convert the series to a df
amount_vs_avg = amount_vs_avg.to_frame(name='amount_over_avg')

#merge to zar_fraud
zar_fraud = zar_fraud.merge(amount_vs_avg, left_index=True, right_index=True)

zar_fraud


,transaction_id,customer_id,timestamp,amount_zar,merchant_category,province,customer_home_province,is_fraud,hour,period_between_transx,cust_avg_amount,amount_over_avg
0,TXN00001,CUST031,2026-01-01 02:24:15,1881.28,Clothing,Gauteng,Gauteng,1,2,NaN,998.13,1.88
1,TXN00002,CUST042,2026-01-02 02:50:18,2212.07,Online Retail,North West,North West,1,2,NaN,1065.60,2.08
2,TXN00003,CUST016,2026-01-02 14:05:14,486.45,Fuel,Mpumalanga,Mpumalanga,0,14,NaN,568.42,0.86
3,TXN00004,CUST017,2026-01-02 16:07:43,2009.66,Travel,Limpopo,Limpopo,0,16,NaN,811.80,2.48
4,TXN00005,CUST025,2026-01-03 11:04:22,619.25,Clothing,Gauteng,Gauteng,0,11,NaN,557.51,1.11
...,...,...,...,...,...,...,...,...,...,...,...,...
355,TXN00356,CUST001,2026-03-30 19:34:02,517.01,Electronics,Gauteng,Gauteng,0,19,43666.0,518.95,1.00
356,TXN00357,CUST027,2026-03-31 07:55:02,273.42,Restaurants,North West,North West,0,7,28111.0,843.84,0.32
357,TXN00358,CUST020,2026-03-31 10:08:12,249.62,Entertainment,Mpumalanga,Mpumalanga,0,10,1068.0,558.36,0.45
358,TXN00359,CUST020,2026-03-31 15:50:23,704.33,Fuel,Mpumalanga,Mpumalanga,0,15,342.0,558.36,1.26


In [13]:
zar_fraud.columns

Index(['transaction_id', 'customer_id', 'timestamp', 'amount_zar',
       'merchant_category', 'province', 'customer_home_province', 'is_fraud',
       'hour', 'period_between_transx', 'cust_avg_amount', 'amount_over_avg'],
      dtype='object')

In [14]:
# compare if customer province is == to merchant province
zar_fraud['is_out_of_province'] = np.where(zar_fraud['customer_home_province'] != zar_fraud['province'], 1, 0)


In [15]:
# transactions happening at midnight
zar_fraud['is_midnight'] = np.where(zar_fraud['hour'].between(1,5), 1, 0)


In [16]:
# Rapid transactions
zar_fraud['rapid_transx'] = np.where(zar_fraud['period_between_transx'].between(0,15),1,0)
zar_fraud.head()

,transaction_id,customer_id,timestamp,amount_zar,merchant_category,province,customer_home_province,is_fraud,hour,period_between_transx,cust_avg_amount,amount_over_avg,is_out_of_province,is_midnight,rapid_transx
0,TXN00001,CUST031,2026-01-01 02:24:15,1881.28,Clothing,Gauteng,Gauteng,1,2,NaN,998.13,1.88,0,1,0
1,TXN00002,CUST042,2026-01-02 02:50:18,2212.07,Online Retail,North West,North West,1,2,NaN,1065.60,2.08,0,1,0
2,TXN00003,CUST016,2026-01-02 14:05:14,486.45,Fuel,Mpumalanga,Mpumalanga,0,14,NaN,568.42,0.86,0,0,0
3,TXN00004,CUST017,2026-01-02 16:07:43,2009.66,Travel,Limpopo,Limpopo,0,16,NaN,811.80,2.48,0,0,0
4,TXN00005,CUST025,2026-01-03 11:04:22,619.25,Clothing,Gauteng,Gauteng,0,11,NaN,557.51,1.11,0,0,0


In [17]:
(zar_fraud['period_between_transx']==0).sum()

np.int64(0)

In [18]:
#list of columns that will be used to build X
feature_cols = ['amount_zar', 'amount_over_avg', 'hour', 'is_midnight', 'is_out_of_province', 'rapid_transx', 'period_between_transx']

X = zar_fraud[feature_cols]
y = zar_fraud['is_fraud']
#

In [19]:
X.head()

,amount_zar,amount_over_avg,hour,is_midnight,is_out_of_province,rapid_transx,period_between_transx
0,1881.28,1.88,2,1,0,0,NaN
1,2212.07,2.08,2,1,0,0,NaN
2,486.45,0.86,14,0,0,0,NaN
3,2009.66,2.48,16,0,0,0,NaN
4,619.25,1.11,11,0,0,0,NaN


In [20]:
#Hadling the NaN to prevent an error
X['period_between_transx'] = X['period_between_transx'].fillna(9999)


/tmp/ipykernel_1965/104354563.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['period_between_transx'] = X['period_between_transx'].fillna(9999)


In [21]:
#import Scikit learn to prepare and train our model
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


#

In [22]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(252, 7) (108, 7)
is_fraud
0    227
1     25
Name: count, dtype: int64
is_fraud
0    97
1    11
Name: count, dtype: int64


In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
X_train_scaled[:5]

array([[ 0.76880998,  1.00455526,  1.69417705, -0.23322372,  7.87400787,
        -0.18107149, -0.62688158],
       [-0.34262615,  0.1524812 , -1.19050279, -0.23322372, -0.12700013,
        -0.18107149, -0.23081482],
       [-0.18976364, -0.24078375, -0.57235711, -0.23322372, -0.12700013,
        -0.18107149,  0.75921018],
       [-0.97719465, -1.28949028, -0.36630855, -0.23322372, -0.12700013,
        -0.18107149, -0.35583625],
       [-0.4203702 , -0.37187206,  0.04578857, -0.23322372, -0.12700013,
        -0.18107149, -0.5217983 ]])

In [25]:
#train the model without including Balance
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)


LogisticRegression()

In [26]:
#Predict on the test
y_pred = model.predict(X_test_scaled)

In [27]:
pd.Series(y_pred).value_counts()

,count
0,100
1,8


In [28]:
#Evaluate how many of the 8 caught were actually correct
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)

print(pd.DataFrame(cm, index = ['Actual: Not Fraud','Actual: Fraud'], columns = ['Predicted: Not Fraud','Predicted: Fraud']))

print()
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))


                   Predicted: Not Fraud  Predicted: Fraud
Actual: Not Fraud                    97                 0
Actual: Fraud                         3                 8

              precision    recall  f1-score   support

   Not Fraud       0.97      1.00      0.98        97
       Fraud       1.00      0.73      0.84        11

    accuracy                           0.97       108
   macro avg       0.98      0.86      0.91       108
weighted avg       0.97      0.97      0.97       108



In [29]:
#Which coefficients mattered most
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})
print(coef_df.sort_values('coefficient', ascending=False).round(2))

                 feature  coefficient
5           rapid_transx         0.90
4     is_out_of_province         0.84
0             amount_zar         0.57
3            is_midnight         0.22
1        amount_over_avg         0.14
6  period_between_transx        -0.06
2                   hour        -0.07


In [30]:
#train the model by including Balance
model_balanced = LogisticRegression(random_state = 42, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)


LogisticRegression(class_weight='balanced', random_state=42)

In [32]:
#prediction on the test
y_pred_balanced = model_balanced.predict(X_test_scaled)

pd.Series.value_counts(y_pred_balanced)



,count
0,89
1,19


In [35]:
#creating the classification report
#2x2 grid
cm_balanced = confusion_matrix(y_test,y_pred_balanced)

#put it in a dataframe
print(pd.DataFrame(cm_balanced, index = ['Actual: Not Fraud','Actual: Fraud'], columns = ['Predicted: Not Fraud','Predicted: Fraud']))

print()
print(classification_report(y_test, y_pred_balanced, target_names=['Not Fraud', 'Fraud']))

                   Predicted: Not Fraud  Predicted: Fraud
Actual: Not Fraud                    88                 9
Actual: Fraud                         1                10

              precision    recall  f1-score   support

   Not Fraud       0.99      0.91      0.95        97
       Fraud       0.53      0.91      0.67        11

    accuracy                           0.91       108
   macro avg       0.76      0.91      0.81       108
weighted avg       0.94      0.91      0.92       108



In [37]:
#which coefficients mattered the most
coef_balanced_df = pd.DataFrame({'feature': X.columns, 'coefficient': model_balanced.coef_[0]})

print(coef_balanced_df.sort_values('coefficient', ascending=False).round(2))

                 feature  coefficient
5           rapid_transx         0.88
4     is_out_of_province         0.78
0             amount_zar         0.39
3            is_midnight         0.27
1        amount_over_avg         0.15
6  period_between_transx         0.00
2                   hour        -0.06
